# Inference: Pizza vs Sushi Classification

This notebook loads the self fine-tuned Vision Transformer (ViT-XS-4) and performs inference on any image.

# 1. Import Libraries

In [37]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchvision import transforms

from PIL import Image
import matplotlib.pyplot as plt
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

Using cpu


# 2. Connect to Google Drive

In [38]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    MODEL_PATH = '/content/drive/MyDrive/DL/FT_ViT_Pizza_Sushi.pth'
    TEST_IMAGE = '/content/drive/MyDrive/DL/sushi.png'
except(Exception) as e:
    print(f"❌ Error: {e}")
    print('Using local')
    MODEL_PATH = './FT_Pizza_Sushi.pth'
    TEST_IMAGE = './sushi.png'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 3. The Model

The Model an the hyperparameters need to be the same with the pretrained one

In [39]:
IMG_SIZE = 64
PATCH_SIZE = 4
IN_CHANNELS = 3
EMBED_DIM = 256
MLP_DIM = 512
ENC_NUMS = 6
MSA_NUMS = 8
CLS_NUMS = 2   # Pizza vs Sushi
DROP_RATE = 0.1

In [40]:
class EmbeddedPatches(nn.Module):
    def __init__(self, img_size, in_channels, embed_dim, patch_size, batch_size):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, patch_size, patch_size)
        N = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_token = nn.Parameter(torch.randn(1, N+1, embed_dim))

    def forward(self, x):
        B = x.size(0)
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        cls_token = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.pos_token
        return x

class MLP(nn.Module):
    def __init__(self, in_features, out_features, drop_rate):
        super().__init__()
        self.layer1 = nn.Linear(in_features, out_features)
        self.layer2 = nn.Linear(out_features, in_features)
        self.dropout = nn.Dropout(drop_rate)

    def forward(self, x):
        x = self.layer1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.layer2(x)
        x = self.dropout(x)
        return x

class VisionEncoder(nn.Module):
    def __init__(self, embed_dim, msa_size, mlp_dim, enc_dim, drop_rate):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, msa_size, drop_rate, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim, mlp_dim, drop_rate)

    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ViT(nn.Module):
    def __init__(self, img_size, patch_size, batch_size, in_channels, embed_dim, enc_dim, msa_size, mlp_dim, cls_nums, drop_rate):
        super().__init__()
        self.embed = EmbeddedPatches(img_size, in_channels, embed_dim, patch_size, batch_size)
        self.encoder = nn.Sequential(*[
            VisionEncoder(embed_dim, msa_size, mlp_dim, enc_dim, drop_rate)
            for _ in range(enc_dim)
        ])
        self.mlp_head = nn.Linear(embed_dim, cls_nums)

        print("✅ Model created")

    def forward(self, x):
        x = self.embed(x)
        x = self.encoder(x)
        cls_token = x[:, 0]
        x = self.mlp_head(cls_token)
        return x

# 3. Load the model

In [41]:
def load_model(weights_path, device):
    # 1. Create the model
    model = ViT(IMG_SIZE, PATCH_SIZE, IMG_SIZE, IN_CHANNELS, 
                EMBED_DIM, ENC_NUMS, MSA_NUMS, MLP_DIM, CLS_NUMS, DROP_RATE)
    
    # 2. Load the pretrained weights
    state_dict = torch.load(weights_path, map_location=device)
    # map_location=device tells PyTorch where to put the tensors as they are
    # being loaded from the file. When a model was saved in PyTorch, the weights
    # remember which device they were on. If we try to load them on a different 
    # hardware, it can run into issues.

    # 3. Load the pretrained weights into the model
    model.load_state_dict(state_dict)

    # 4. Move the model to device and put it in eval mode
    model.to(device)
    model.eval()

    print("✅ Custom Model loaded")

    return model

model = load_model(MODEL_PATH, device)

✅ Model created
✅ Custom Model loaded


# 4. Transform Input Image

In [42]:
class LetterBox:
    """
    - Pads an image to a square canvas with a neural gray color
    - Placing the image in the center.
    - Portrait images get horizontal bars (top & bottom)
    - Landscape images get vertical bars (left & right)
    """

    def __init__(self, fill_color=(128, 128, 128)):
        self.fill_color = fill_color
    
    def __call__(self, img):
        w, h = img.size # PIL uses (width, height)

        side = max(w, h) # square canvas size

        # Create a neutral gray square canvas
        canvas = Image.new('RGB', (side, side), self.fill_color)

        # Paste the original image centered on the canvas
        pad_left = (side - w) // 2
        pad_top = (side - h) // 2
        canvas.paste(img, (pad_left, pad_top))

        return canvas


In [43]:
# # Standard ImageNet normalization used during training
stats = ((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))

inference_transform = transforms.Compose([
    LetterBox(fill_color=(128, 128, 128)),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

# 5. Predict Function

In [50]:
def predict(image_path, model, transform, classes=['Pizza', 'Sushi']):
    # 1. Load the Image and convert to RGB
    img = Image.open(image_path).convert('RGB')

    # 2. Transform the image and turn into 4D-Tensor
    # (C, H, W) -> (B, C, H, W)
    input_tensor = transform(img).unsqueeze(0).to(device)

    # 3. Execute Inference
    with torch.no_grad():
        output = model(input_tensor)
        probs = F.softmax(output, dim=1)
        print(probs)

In [51]:
predict(TEST_IMAGE, model, inference_transform)

tensor([[0.0028, 0.9972]])
